# NFL Combine 40-Yard Dash Movement Time Series — quick start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/phcs971/nfl-combine-40-timeseries/blob/main/notebooks/quickstart.ipynb)

Loads the dataset straight from GitHub (no clone needed), shows its structure, plots one run and the class medians, and trains a simple baseline. Data licence CC BY 4.0; see the repository README and DATASHEET for how it was built.

In [ ]:
import pandas as pd

# "main" is the latest data; use a release tag such as "v1.0.0" to pin a version.
VERSION = "main"
BASE = f"https://raw.githubusercontent.com/phcs971/nfl-combine-40-timeseries/{VERSION}/data"

runs = pd.read_csv(f"{BASE}/runs.csv")
series = pd.read_parquet(f"{BASE}/series.parquet")               # one row per video frame
by_time = pd.read_parquet(f"{BASE}/series_by_time.parquet")      # 101 points per run
by_distance = pd.read_parquet(f"{BASE}/series_by_distance.parquet")  # every 0.25 yd, 0-39 yd

ok = runs[runs.status == "ok"]
print(runs.status.value_counts().to_dict())
ok.groupby(["cls", "split"]).agg(runs=("run_id", "size"), athletes=("athlete", "nunique"))

In [ ]:
# Save local copies, e.g. to use offline or upload elsewhere.
import os
os.makedirs("nfl40", exist_ok=True)
for name, df in [("runs.csv", runs), ("series.parquet", series),
                 ("series_by_time.parquet", by_time), ("series_by_distance.parquet", by_distance)]:
    path = os.path.join("nfl40", name)
    df.to_csv(path, index=False) if name.endswith(".csv") else df.to_parquet(path, index=False)
sorted(os.listdir("nfl40"))

In [ ]:
# One run, frame by frame. time_40yd is the unofficial on-screen time.
import matplotlib.pyplot as plt

run = ok.iloc[0]
s = series[series.run_id == run.run_id]
cols = ["x_yd", "v_yds", "trunk_angle", "hip_height_ratio", "knee_lead"]
fig, axes = plt.subplots(len(cols), 1, figsize=(9, 8), sharex=True)
for ax, c in zip(axes, cols):
    ax.plot(s.t_clock, s[c])
    ax.set_ylabel(c)
axes[-1].set_xlabel("t_clock (s)")
fig.suptitle(f"{run.run_id} ({run.position}, {run.cls}), 40-yd time {run.time_40yd:.2f} s")
plt.show()

In [ ]:
# Class medians along the run (distance grid).
d = by_distance.merge(ok[["run_id", "cls"]], on="run_id")
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, c in zip(axes, ["v_yds", "trunk_angle", "hip_height_ratio"]):
    for cls, g in d.groupby("cls"):
        ax.plot(g.groupby("x_yd")[c].median(), label=cls)
    ax.set_title(c)
    ax.set_xlabel("distance (yd)")
axes[0].legend()
plt.show()

In [ ]:
# Baseline: logistic regression on the equal-length time grid, using the given athlete split.
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

channels = ["v_yds", "trunk_angle", "hip_height_ratio"]
t = by_time.merge(ok[["run_id", "cls", "split"]], on="run_id")
X = t.pivot(index="run_id", columns="phase", values=channels)
X = X.T.interpolate(limit_direction="both").T  # fill the rare gaps along each run
meta = ok.set_index("run_id").loc[X.index]
train, test = meta.split == "train", meta.split == "test"
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.1))
model.fit(X[train.values], meta.cls[train])
print("test accuracy:", round(model.score(X[test.values], meta.cls[test]), 3))